# HTD OOF v2 Qwen3-VL End-to-End + YOLO Submission

Thunder Compute/A6000 inference notebook. Place this file next to `HTD_OOF_v2_Qwen3VL8B_end_to_end.ipynb` and run it from the same directory, or set `HTD_NOTEBOOK_DIR` to this folder.

It uses DocLayout-YOLO `DoclayoutYoloV4.1.pt` for `bbox` + `type`, then uses the OOF v2 Qwen3-VL end-to-end LoRA adapters as crop OCR models.

Expected output: `submission.csv` with columns `image,regions`.


In [ ]:
# =============================================================================
# USER CONFIG - EDIT THIS CELL ONLY
# =============================================================================
# Empty strings mean "use the notebook default". Paths are resolved by later cells.
import os

USER_CONFIG = {
    # Core local paths. Keep HTD_NOTEBOOK_DIR="." when you launch Jupyter from OOF/v2.
    "HTD_NOTEBOOK_DIR": ".",
    "HTD_MODEL_DOWNLOAD_DIR": "",  # default: NOTEBOOK_DIR/models/Qwen3-VL-8B-Instruct
    "HTD_DATASET_DOWNLOAD_DIR": "",  # default: NOTEBOOK_DIR/data/rukopys-dataset
    "HTD_DATASET_ROOT": "",  # set only if dataset is unpacked somewhere else
    "HTD_OUTPUT_ROOT": "",  # default: NOTEBOOK_DIR/outputs/htd_oof_v2_qwen3vl_end_to_end
    "HTD_OUTPUT_CSV": "",  # default: NOTEBOOK_DIR/submission.csv

    # Dependency/install controls for Thunder Compute.
    "HTD_INSTALL_DEPS": True,
    "HTD_INSTALL_DOCLAYOUT_YOLO": True,
    "HTD_DOCLAYOUT_REPO": "",  # default: NOTEBOOK_DIR/DocLayout-YOLO

    # YOLO weights. Put DoclayoutYoloV4.1.pt in the same folder as this notebook, or set this path.
    "HTD_YOLO_WEIGHTS_PATH": "",
    "HTD_DOWNLOAD_YOLO_WITH_KAGGLEHUB": False,

    # Run split/output.
    "HTD_RUN_SPLIT": "test",  # choices: test, validation
    "HTD_VALIDATION_JSONL": "",  # required only when HTD_RUN_SPLIT="validation"
    "HTD_SUBMIT_RUN_NAME": "oof_v2_e2e_yolo_v41_a6000",
    "HTD_RESUME_PARTIALS": True,

    # Thunder/A6000 runtime. Leave model/dtype empty for auto BF16 on A6000, FP16 otherwise.
    "HTD_REQUESTED_NUM_GPUS": 1,
    "HTD_REQUIRE_REQUESTED_GPUS": False,
    "HTD_MODEL_LOAD_MODE": "",  # auto; set fp16/bf16/4bit only if needed
    "HTD_INFERENCE_DTYPE": "",  # auto; set float16/bfloat16 only if needed

    # Quick smoke test.
    "HTD_TEST_MODE": False,
    "HTD_TEST_MAX_SAMPLES": 4,

    # LoRA ensemble/OCR.
    "HTD_MODEL_IDS": "1",  # set "1", "2", "3", or e.g. "1,3" to choose LoRA models
    "HTD_USE_ALL_LORA_ADAPTERS": True,
    "HTD_MAX_LORA_ADAPTERS": 3,
    "HTD_ENSEMBLE_STRATEGY": "consensus",  # choices: consensus, first
    "HTD_CROP_BATCH_SIZE": 32,  # 8/16 is usually faster than 32 for Qwen-VL crop batches
    "HTD_MAX_PIXELS_CROP": 262_144,
    "HTD_MAX_NEW_TOKENS_CROP": 256,
    "HTD_MAX_CROP_TEXT_CHARS": 1500,
    "HTD_CROP_PAD_RATIO": 0.04,
    "HTD_CROP_OCR_MODE": "all_text",  # choices: none, smart, all_text
    "HTD_SORT_CROPS_BY_AREA": True,
    "HTD_CLEAR_CUDA_CACHE_EVERY_BATCH": False,

    # YOLO detection knobs. Usually do not change unless you are reproducing another YOLO config.
    "HTD_YOLO_IMG_SIZE": 1280,
    "HTD_YOLO_CONF": 0.20,
    "HTD_YOLO_MAX_DET": 220,
    "HTD_YOLO_IOU_NMS": 0.60,
    "HTD_YOLO_DEDUP_IOU": 0.90,
    "HTD_YOLO_PAD_SCALE_X": 0.00,
    "HTD_YOLO_PAD_SCALE_Y": 0.00,

    # Logging/checkpointing.
    "HTD_CHECKPOINT_EVERY": 10,
    "HTD_PROGRESS_LOG_EVERY": 1,
    "HTD_PROGRESS_BAR_WIDTH": 20,
    "HTD_USE_TQDM_PROGRESS": False,
    "HTD_PROFILE_OCR": True,  # print per-image/per-crop-batch timing to find bottlenecks
}


def apply_user_config(config):
    applied = []
    skipped_blank = []
    for key, value in config.items():
        if value is None:
            continue
        if isinstance(value, bool):
            os.environ[key] = "1" if value else "0"
        elif isinstance(value, str) and value.strip() == "":
            skipped_blank.append(key)
            continue
        else:
            os.environ[key] = str(value)
        applied.append(key)
    print("Applied USER_CONFIG keys:", len(applied))
    print("Blank/default keys:", ", ".join(skipped_blank) if skipped_blank else "none")


apply_user_config(USER_CONFIG)


In [ ]:
# Thunder Compute / A6000 dependency cell.
# Values come from the USER CONFIG cell above. Set HTD_INSTALL_DEPS=False there if packages are already installed.
import os
from pathlib import Path

NOTEBOOK_DIR_FOR_INSTALL = Path(os.environ.get("HTD_NOTEBOOK_DIR", ".")).expanduser().resolve()
INSTALL_DEPS = bool(int(os.environ.get("HTD_INSTALL_DEPS", "1")))
INSTALL_DOCLAYOUT_YOLO = bool(int(os.environ.get("HTD_INSTALL_DOCLAYOUT_YOLO", "1")))
DOCLAYOUT_REPO = Path(
    os.environ.get("HTD_DOCLAYOUT_REPO", str(NOTEBOOK_DIR_FOR_INSTALL / "DocLayout-YOLO"))
).expanduser().resolve()


def run_quiet(cmd, check=True):
    import subprocess

    print("Running:", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, check=check)


def install_system_libs_for_cv2():
    import shutil
    import subprocess
    import sys

    apt_get = shutil.which("apt-get")
    if not apt_get:
        print("apt-get not found; skipping system OpenCV libs.", flush=True)
        return
    base_cmd = [apt_get]
    if os.geteuid() != 0:
        sudo = shutil.which("sudo")
        if not sudo:
            print("No root/sudo; skipping system OpenCV libs. Headless OpenCV fallback will be used.", flush=True)
            return
        base_cmd = [sudo, apt_get]
    packages = ["libxcb1", "libgl1", "libglib2.0-0"]
    try:
        subprocess.run(base_cmd + ["update", "-qq"], check=False)
        subprocess.run(base_cmd + ["install", "-y", "-qq", *packages], check=False)
    except Exception as e:
        print("System OpenCV lib install skipped/failed:", e, flush=True)


def force_headless_opencv():
    import subprocess
    import sys

    # DocLayout-YOLO/ultralytics may pull GUI OpenCV. Remove it and put headless last.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "opencv-python", "opencv-contrib-python"],
        check=False,
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "opencv-python-headless"])

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "qwen-vl-utils",
            "huggingface_hub",
            "hf_transfer",
            "kagglehub",
            "opencv-python-headless",
            "pillow<12",
            "torchvision",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)

if INSTALL_DOCLAYOUT_YOLO:
    import subprocess
    import sys

    repo_candidates = [
        DOCLAYOUT_REPO,
        NOTEBOOK_DIR_FOR_INSTALL / "DocLayout-YOLO",
        Path("/kaggle/input/DocLayout-YOLO"),
        Path("/kaggle/input/doclayout-yolo/DocLayout-YOLO"),
    ]
    repo = next((candidate for candidate in repo_candidates if candidate.exists()), DOCLAYOUT_REPO)
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/opendatalab/DocLayout-YOLO.git", str(repo)])
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    # Thunder uses a writable local clone such as ~/DocLayout-YOLO; install it editable so deps like cv2 are available.
    # Skip only read-only Kaggle input mounts.
    if not str(repo).startswith("/kaggle/input"):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)])
    force_headless_opencv()


In [ ]:
import gc
import json
import logging
import math
import os
import re
import subprocess
import sys
import time
import warnings
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from types import ModuleType

import pandas as pd
import torch
from PIL import Image
from torchvision.ops import nms
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


def env_bool(name, default):
    return bool(int(os.environ.get(name, "1" if default else "0")))


def env_int(name, default):
    return int(os.environ.get(name, str(default)))


def parse_model_ids(value):
    text = str(value or "1,2,3").strip().lower()
    if text in {"all", "*"}:
        return [1, 2, 3]
    ids = []
    for part in re.split(r"[,\s]+", text):
        if not part:
            continue
        mid = int(part)
        if mid not in {1, 2, 3}:
            raise ValueError(f"HTD_MODEL_IDS only supports 1, 2, 3; got {mid}")
        if mid not in ids:
            ids.append(mid)
    if not ids:
        raise ValueError("HTD_MODEL_IDS resolved to no model ids")
    return ids


def suppress_transformers_noise():
    message = r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*"
    warnings.filterwarnings("ignore", message=message)
    logging.getLogger("transformers").setLevel(logging.ERROR)
    logging.getLogger("transformers.processing_utils").setLevel(logging.ERROR)
    try:
        from transformers.utils import logging as hf_logging

        hf_logging.set_verbosity_error()
    except Exception:
        pass


suppress_transformers_noise()

for repo_path in [
    str(Path(os.environ.get("HTD_DOCLAYOUT_REPO", "")).expanduser()) if os.environ.get("HTD_DOCLAYOUT_REPO") else "",
    str(Path(os.environ.get("HTD_NOTEBOOK_DIR", ".")).expanduser().resolve() / "DocLayout-YOLO"),
    "/kaggle/working/DocLayout-YOLO",
    "/kaggle/input/DocLayout-YOLO",
    "/kaggle/input/doclayout-yolo/DocLayout-YOLO",
]:
    if not repo_path:
        continue
    p = Path(repo_path)
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

from doclayout_yolo import YOLOv10

# =============================================================================
# Thunder/A6000 config. These defaults mirror the successful train notebook layout.
# =============================================================================
NOTEBOOK_DIR = Path(os.environ.get("HTD_NOTEBOOK_DIR", ".")).expanduser().resolve()
RUN_NAME = os.environ.get("HTD_SUBMIT_RUN_NAME", "oof_v2_e2e_yolo_v41_a6000")

MODEL_DOWNLOAD_DIR = Path(
    os.environ.get("HTD_MODEL_DOWNLOAD_DIR", str(NOTEBOOK_DIR / "models" / "Qwen3-VL-8B-Instruct"))
).expanduser().resolve()
DATASET_ROOT_OVERRIDE = os.environ.get("HTD_DATASET_ROOT", "").strip()
DATASET_DOWNLOAD_DIR = Path(
    os.environ.get("HTD_DATASET_DOWNLOAD_DIR", str(NOTEBOOK_DIR / "data" / "rukopys-dataset"))
).expanduser().resolve()
OUTPUT_ROOT = Path(
    os.environ.get("HTD_OUTPUT_ROOT", str(NOTEBOOK_DIR / "outputs" / "htd_oof_v2_qwen3vl_end_to_end"))
).expanduser().resolve()
YOLO_WEIGHTS_PATH = os.environ.get("HTD_YOLO_WEIGHTS_PATH", "").strip()

BASE_MODEL_CANDIDATES = [
    MODEL_DOWNLOAD_DIR,
    NOTEBOOK_DIR / "Qwen3-VL-8B-Instruct",
    NOTEBOOK_DIR / "qwen3-vl-8b-instruct",
    Path("/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1"),
    Path("/kaggle/input/qwen3-vl-8b-instruct"),
    "Qwen/Qwen3-VL-8B-Instruct",
]

DATASET_ROOT_CANDIDATES = [
    DATASET_DOWNLOAD_DIR,
    NOTEBOOK_DIR / "data" / "rukopys-dataset",
    Path("/kaggle/input/datasets/quii29/rukopys-dataset"),
    Path("/kaggle/input/rukopys-dataset"),
]
if DATASET_ROOT_OVERRIDE:
    DATASET_ROOT_CANDIDATES.insert(0, Path(DATASET_ROOT_OVERRIDE).expanduser().resolve())

RUN_SPLIT = os.environ.get("HTD_RUN_SPLIT", "test")
VALIDATION_JSONL = os.environ.get("HTD_VALIDATION_JSONL", "").strip()

MODEL_RUNS = {
    1: "model1_train_folds1_2_heldout_fold3",
    2: "model2_train_folds1_3_heldout_fold2",
    3: "model3_train_folds2_3_heldout_fold1",
}
SELECTED_MODEL_IDS = parse_model_ids(os.environ.get("HTD_MODEL_IDS", "1,2,3"))
OOF_OUTPUT_ROOTS = [
    OUTPUT_ROOT,
    NOTEBOOK_DIR / "outputs" / "htd_oof_v2_qwen3vl_end_to_end",
    Path("/kaggle/working/outputs/htd_oof_v2_qwen3vl_end_to_end"),
]
LORA_CANDIDATES = []
for root in OOF_OUTPUT_ROOTS:
    for model_id_for_path in SELECTED_MODEL_IDS:
        run = MODEL_RUNS[model_id_for_path]
        LORA_CANDIDATES.append(root / run / f"qwen3vl_oof_v2_e2e_{run}_lora_final")

YOLO_WEIGHT_CANDIDATES = []
if YOLO_WEIGHTS_PATH:
    YOLO_WEIGHT_CANDIDATES.append(Path(YOLO_WEIGHTS_PATH).expanduser().resolve())
YOLO_WEIGHT_CANDIDATES.extend([
    NOTEBOOK_DIR / "DoclayoutYoloV4.1.pt",
    NOTEBOOK_DIR / "models" / "DoclayoutYoloV4.1.pt",
    NOTEBOOK_DIR / "yolo" / "DoclayoutYoloV4.1.pt",
    Path("/kaggle/working/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/1/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/2/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/3/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/4/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/htd-box-doclayoutyolo-v4/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/doclayoutyolov4-1/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/doclayout-yolo-v4-1/DoclayoutYoloV4.1.pt"),
])
YOLO_KAGGLEHUB_HANDLES = [
    "notpitomon/htd-box-doclayoutyolo-v4/pytorch/default",
    "notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/4",
    "notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/3",
    "notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/2",
    "notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/1",
]
DOWNLOAD_YOLO_WITH_KAGGLEHUB = env_bool("HTD_DOWNLOAD_YOLO_WITH_KAGGLEHUB", False)

REQUESTED_NUM_GPUS = env_int("HTD_REQUESTED_NUM_GPUS", 1)
REQUIRE_REQUESTED_GPUS = env_bool("HTD_REQUIRE_REQUESTED_GPUS", False)
TEST_MODE = env_bool("HTD_TEST_MODE", False)
TEST_MAX_SAMPLES = env_int("HTD_TEST_MAX_SAMPLES", 4)

CUDA_MAJOR = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
USE_BF16 = CUDA_MAJOR >= 8
MODEL_LOAD_MODE = os.environ.get("HTD_MODEL_LOAD_MODE", "bf16" if USE_BF16 else "fp16").strip().lower()
INFERENCE_DTYPE_NAME = os.environ.get("HTD_INFERENCE_DTYPE", "bfloat16" if USE_BF16 else "float16").strip().lower()
TORCH_DTYPE = torch.bfloat16 if INFERENCE_DTYPE_NAME in {"bf16", "bfloat16"} else torch.float16

USE_ALL_LORA_ADAPTERS = env_bool("HTD_USE_ALL_LORA_ADAPTERS", True)
MAX_LORA_ADAPTERS = env_int("HTD_MAX_LORA_ADAPTERS", 3)
ENSEMBLE_STRATEGY = os.environ.get("HTD_ENSEMBLE_STRATEGY", "consensus")
CROP_BATCH_SIZE = env_int("HTD_CROP_BATCH_SIZE", 8)
MAX_PIXELS_CROP = env_int("HTD_MAX_PIXELS_CROP", 262_144)
MAX_NEW_TOKENS_CROP = env_int("HTD_MAX_NEW_TOKENS_CROP", 256)
MAX_CROP_TEXT_CHARS = env_int("HTD_MAX_CROP_TEXT_CHARS", 1500)
CROP_PAD_RATIO = float(os.environ.get("HTD_CROP_PAD_RATIO", "0.04"))
CROP_OCR_MODE = os.environ.get("HTD_CROP_OCR_MODE", "all_text")
SORT_CROPS_BY_AREA = env_bool("HTD_SORT_CROPS_BY_AREA", True)
CLEAR_CUDA_CACHE_EVERY_BATCH = env_bool("HTD_CLEAR_CUDA_CACHE_EVERY_BATCH", False)

YOLO_IMG_SIZE = env_int("HTD_YOLO_IMG_SIZE", 1280)
YOLO_CONF = float(os.environ.get("HTD_YOLO_CONF", "0.20"))
YOLO_MAX_DET = env_int("HTD_YOLO_MAX_DET", 220)
YOLO_IOU_NMS = float(os.environ.get("HTD_YOLO_IOU_NMS", "0.60"))
YOLO_DEDUP_IOU = float(os.environ.get("HTD_YOLO_DEDUP_IOU", "0.90"))
YOLO_PAD_SCALE_X = float(os.environ.get("HTD_YOLO_PAD_SCALE_X", "0.00"))
YOLO_PAD_SCALE_Y = float(os.environ.get("HTD_YOLO_PAD_SCALE_Y", "0.00"))

WORK_DIR = NOTEBOOK_DIR / f"{RUN_NAME}_work"
PARTIAL_DIR = WORK_DIR / "partials"
OUTPUT_CSV = Path(os.environ.get("HTD_OUTPUT_CSV", str(NOTEBOOK_DIR / "submission.csv"))).expanduser().resolve()
RESUME_PARTIALS = env_bool("HTD_RESUME_PARTIALS", True)
CHECKPOINT_EVERY = env_int("HTD_CHECKPOINT_EVERY", 10)
PROGRESS_LOG_EVERY = env_int("HTD_PROGRESS_LOG_EVERY", 1)
PROGRESS_BAR_WIDTH = env_int("HTD_PROGRESS_BAR_WIDTH", 20)
USE_TQDM_PROGRESS = env_bool("HTD_USE_TQDM_PROGRESS", False)
PROFILE_OCR = env_bool("HTD_PROFILE_OCR", True)
HYBRID_PARTIAL_PREFIX = str(PARTIAL_DIR / f"{RUN_NAME}_partial_results_gpu")
for path in [WORK_DIR, PARTIAL_DIR, OUTPUT_CSV.parent]:
    path.mkdir(parents=True, exist_ok=True)

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
TEXT_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"]

print("Notebook dir:", NOTEBOOK_DIR)
print("Model download dir:", MODEL_DOWNLOAD_DIR)
print("Dataset download dir:", DATASET_DOWNLOAD_DIR)
print("Output root:", OUTPUT_ROOT)
print("Output CSV:", OUTPUT_CSV)
print("CUDA major:", CUDA_MAJOR, "bf16:", USE_BF16)
print("Model load mode:", MODEL_LOAD_MODE, "dtype:", TORCH_DTYPE)
print("Selected model IDs:", SELECTED_MODEL_IDS)
print("Requested GPUs:", REQUESTED_NUM_GPUS)


In [ ]:
def format_bytes(num_bytes):
    try:
        value = float(num_bytes)
    except Exception:
        return "unknown"
    units = ["B", "KB", "MB", "GB", "TB"]
    idx = 0
    while value >= 1024 and idx < len(units) - 1:
        value /= 1024.0
        idx += 1
    return f"{value:.1f}{units[idx]}"


def path_tree_stats(path):
    path = Path(path)
    if path.is_file():
        return 1, path.stat().st_size
    files = 0
    total = 0
    if not path.exists():
        return 0, 0
    for item in path.rglob("*"):
        if item.is_file():
            files += 1
            try:
                total += item.stat().st_size
            except OSError:
                pass
    return files, total


def log_path_stats(label, path):
    if isinstance(path, str) and not path.startswith("/"):
        print(f"{label}: {path} (hub id)", flush=True)
        return
    p = Path(path)
    files, total = path_tree_stats(p)
    print(f"{label}: {p} exists={p.exists()} files={files:,} size={format_bytes(total)}", flush=True)


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_dataset_root_candidate(path, split):
    path = Path(path)
    if (path / split / "metadata.jsonl").exists() and (path / split / "images").exists():
        return path
    if (path / "metadata.jsonl").exists() and (path / "images").exists() and path.name == split:
        return path.parent
    if path.exists():
        for meta in sorted(path.rglob(f"{split}/metadata.jsonl")):
            root = meta.parent.parent
            if (root / split / "images").exists():
                return root
    return None


def get_dataset_root():
    split = "train" if RUN_SPLIT == "validation" else "test"
    checked = []
    for item in DATASET_ROOT_CANDIDATES:
        checked.append(str(item))
        root = resolve_dataset_root_candidate(item, split)
        if root is not None:
            return root
    hint = "\n".join(checked)
    raise FileNotFoundError(f"Could not find RUKOPYS dataset root for split={split}. Checked:\n{hint}")


def resolve_image_path(root, split, file_name):
    root = Path(root)
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [root / split / file_name, root / file_name]
    for candidate_name in candidate_names:
        candidates.extend([root / split / "images" / candidate_name, root / split / candidate_name])
    for p in candidates:
        if p.exists():
            return str(p)
    return str(root / split / "images" / candidate_names[0])


def find_base_model_id():
    checked = []
    for item in BASE_MODEL_CANDIDATES:
        if isinstance(item, str) and not item.startswith("/"):
            return item
        p = Path(item)
        checked.append(str(p))
        if (p / "config.json").exists():
            return str(p)
    hint = "\n".join(checked)
    raise FileNotFoundError(f"No Qwen3-VL base model found. Checked:\n{hint}")


def has_lora_weights(path):
    path = Path(path)
    return (path / "adapter_config.json").exists() and (
        (path / "adapter_model.safetensors").exists() or (path / "adapter_model.bin").exists()
    )


def scan_oof_v2_lora_dirs():
    roots = [NOTEBOOK_DIR, OUTPUT_ROOT, NOTEBOOK_DIR / "outputs", Path("/kaggle/working"), Path("/kaggle/input")]
    found = []
    for root in roots:
        if not root.exists():
            continue
        for cfg in root.rglob("adapter_config.json"):
            p = cfg.parent
            text = str(p).lower().replace("-", "_")
            if "trainer_checkpoints" in text or "checkpoint_" in text or "checkpoint-" in text:
                continue
            if "oof_v2" in text and ("e2e" in text or "end_to_end" in text):
                if has_lora_weights(p):
                    found.append(p)
    return found


def model_sort_key(path):
    text = str(path).lower()
    match = re.search(r"model(\d+)", text)
    return (int(match.group(1)) if match else 999, text)


def expected_lora_count():
    if not USE_ALL_LORA_ADAPTERS:
        return 1
    return min(MAX_LORA_ADAPTERS, len(SELECTED_MODEL_IDS))


def find_lora_dirs():
    target_count = expected_lora_count()
    found = []
    seen = set()
    for item in LORA_CANDIDATES:
        p = Path(item)
        if has_lora_weights(p) and str(p) not in seen:
            found.append(p)
            seen.add(str(p))
    found = sorted(found, key=model_sort_key)
    if len(found) >= target_count:
        return found[:target_count]

    # Fallback only if final LoRA dirs were not found; do not pull trainer checkpoints into the ensemble.
    for p in scan_oof_v2_lora_dirs():
        if str(p) not in seen:
            found.append(p)
            seen.add(str(p))
    found = sorted(found, key=model_sort_key)
    if not found:
        checked = "\n".join(str(p) for p in LORA_CANDIDATES[:12])
        raise FileNotFoundError(f"No OOF v2 end-to-end LoRA adapters found. Checked primary paths like:\n{checked}")
    if len(found) < target_count:
        print(f"Warning: expected {target_count} final LoRA adapters from HTD_MODEL_IDS={SELECTED_MODEL_IDS}, found {len(found)}.", flush=True)
    return found[:target_count]


def find_doclayout_v41_under(root):
    root = Path(root)
    if root.is_file() and root.name == "DoclayoutYoloV4.1.pt":
        return root
    if root.exists() and root.is_dir():
        exact = sorted(root.rglob("DoclayoutYoloV4.1.pt"))
        if exact:
            return exact[0]
    return None


def download_yolo_weights_with_kagglehub():
    if not DOWNLOAD_YOLO_WITH_KAGGLEHUB:
        return None
    try:
        import kagglehub
    except Exception as e:
        print("kagglehub is not available; skipping YOLO download:", e, flush=True)
        return None

    for handle in YOLO_KAGGLEHUB_HANDLES:
        try:
            print("Downloading/searching YOLO Kaggle model via kagglehub:", handle, flush=True)
            model_dir = Path(kagglehub.model_download(handle))
            found = find_doclayout_v41_under(model_dir)
            if found is not None:
                return found
        except Exception as e:
            print(f"kagglehub model_download failed for {handle}: {e}", flush=True)
    return None


def load_lora_prompt_configs(lora_paths):
    configs = []
    for lora_path in lora_paths:
        cfg_path = Path(lora_path) / "rukopys_prompt_config.json"
        if not cfg_path.exists():
            continue
        try:
            cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
            cfg["_prompt_config_path"] = str(cfg_path)
            cfg["_lora_dir"] = str(lora_path)
            configs.append(cfg)
        except Exception as e:
            print(f"Could not read prompt config {cfg_path}: {e}", flush=True)
    return configs


def select_training_page_prompt(prompt_configs):
    for cfg in prompt_configs:
        page_prompt = str(cfg.get("page_prompt") or "").strip()
        if page_prompt:
            return {
                "prompt_version": cfg.get("prompt_version", "unknown"),
                "page_prompt": page_prompt,
                "source": cfg.get("_prompt_config_path", "adapter prompt config"),
            }
    return {"prompt_version": "missing", "page_prompt": "", "source": "missing"}


def find_yolo_weights():
    checked = []
    for item in YOLO_WEIGHT_CANDIDATES:
        p = Path(item)
        checked.append(str(p))
        found = find_doclayout_v41_under(p)
        if found is not None:
            return found
    for root in [NOTEBOOK_DIR, NOTEBOOK_DIR / "models", NOTEBOOK_DIR / "yolo", Path("/kaggle/working"), Path("/kaggle/input")]:
        found = find_doclayout_v41_under(root)
        if found is not None:
            return found
    downloaded = download_yolo_weights_with_kagglehub()
    if downloaded is not None:
        return downloaded
    hint = "\n".join(checked)
    raise FileNotFoundError(f"No DoclayoutYoloV4.1.pt found. Put it in NOTEBOOK_DIR/models, set HTD_YOLO_WEIGHTS_PATH, or enable kagglehub download. Checked:\n{hint}")


model_id = find_base_model_id()
lora_dirs = find_lora_dirs()
PROMPT_CONFIGS = load_lora_prompt_configs(lora_dirs)
TRAIN_PROMPT = select_training_page_prompt(PROMPT_CONFIGS)
TRAIN_PROMPT_VERSION = TRAIN_PROMPT["prompt_version"]
TRAIN_PAGE_PROMPT = TRAIN_PROMPT["page_prompt"]
TRAIN_PROMPT_SOURCE = TRAIN_PROMPT["source"]
yolo_weights = find_yolo_weights()
dataset_root = get_dataset_root()

if RUN_SPLIT == "validation":
    if not VALIDATION_JSONL:
        raise ValueError("Set VALIDATION_JSONL when RUN_SPLIT='validation'.")
    test_records = read_jsonl(VALIDATION_JSONL)
    IMAGE_SPLIT = "train"
else:
    test_records = read_jsonl(dataset_root / "test" / "metadata.jsonl")
    IMAGE_SPLIT = "test"

if TEST_MODE:
    test_records = test_records[:TEST_MAX_SAMPLES]

log_path_stats("Base model", model_id)
for idx, path in enumerate(lora_dirs, start=1):
    log_path_stats(f"LoRA {idx}", path)
log_path_stats("YOLO weights", yolo_weights)
log_path_stats("Dataset root", dataset_root)
print("Run split:", RUN_SPLIT)
print("Image split:", IMAGE_SPLIT)
print("Images:", len(test_records))
print("LoRA adapters selected:", len(lora_dirs))
print("Prompt configs loaded:", len(PROMPT_CONFIGS))
print("Training prompt version:", TRAIN_PROMPT_VERSION)
print("Training prompt source:", TRAIN_PROMPT_SOURCE)



In [ ]:
YOLO_TYPE_ALIASES = {
    "text": "printed",
    "plain text": "printed",
    "paragraph": "printed",
    "title": "printed",
    "section header": "printed",
    "section-header": "printed",
    "page header": "printed",
    "page-header": "printed",
    "page footer": "printed",
    "page-footer": "printed",
    "caption": "annotation",
    "footnote": "annotation",
    "list item": "printed",
    "list-item": "printed",
    "equation": "formula",
    "math": "formula",
    "picture": "image",
    "figure": "image",
    "fig": "image",
    "diagram": "image",
    "graphic": "graph",
    "chart": "graph",
    "plot": "graph",
}


def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    value = re.sub(r"[_-]+", " ", value)
    value = YOLO_TYPE_ALIASES.get(value, value)
    return value if value in VALID_TYPES else "handwritten"


def clamp_xyxy(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(1, area_a + area_b - inter)


def sort_regions(regions):
    return sorted(regions, key=lambda r: (r["bbox"][1], r["bbox"][0]))


def strip_internal_fields(region):
    return {"bbox": region["bbox"], "type": normalize_type(region.get("type")), "text": str(region.get("text") or "")}


def extract_json_text(obj):
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        if "text" in obj:
            return str(obj.get("text") or "")
        parts = [extract_json_text(v) for v in obj.values()]
        return "\n".join(p for p in parts if p.strip())
    if isinstance(obj, list):
        parts = [extract_json_text(item) for item in obj]
        return "\n".join(p for p in parts if p.strip())
    return ""


def json_loads_relaxed(text):
    try:
        return json.loads(text)
    except Exception:
        pass
    for pattern in [r"\[.*\]", r"\{.*\}"]:
        match = re.search(pattern, text, flags=re.S)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                continue
    return None


def clean_crop_text(text):
    text = str(text or "").strip()
    if not text:
        return ""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    text = text.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_-]*", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    parsed = json_loads_relaxed(text)
    if parsed is not None:
        text = extract_json_text(parsed).strip()
    elif '"text"' in text:
        matches = re.findall(r'"text"\s*:\s*"((?:\\.|[^"\\])*)"', text, flags=re.S)
        decoded = []
        for item in matches:
            try:
                decoded.append(json.loads('"' + item + '"'))
            except Exception:
                decoded.append(item)
        if decoded:
            text = "\n".join(str(x) for x in decoded).strip()

    text = re.sub(r"^(assistant|text|transcription|answer)\s*:\s*", "", text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text[:MAX_CROP_TEXT_CHARS]


def text_quality_penalty(text):
    text = str(text or "")
    if not text.strip():
        return 10.0
    penalty = 0.0
    lower = text.lower()
    bad_markers = ["```", "json", "bbox", "markdown", "explanation", "i cannot", "sorry"]
    penalty += 0.15 * sum(marker in lower for marker in bad_markers)
    if len(text) > MAX_CROP_TEXT_CHARS * 0.95:
        penalty += 0.2
    if re.search(r"(.)\1{12,}", text):
        penalty += 0.2
    return penalty


def choose_consensus_text(candidates):
    cleaned = [clean_crop_text(x) for x in candidates]
    cleaned = [x for x in cleaned if x.strip()]
    if not cleaned:
        return ""
    counts = Counter(cleaned)
    winner, count = counts.most_common(1)[0]
    if count >= 2 or len(cleaned) == 1 or ENSEMBLE_STRATEGY == "first":
        return winner

    best_text = cleaned[0]
    best_score = float("inf")
    for i, cand in enumerate(cleaned):
        distances = []
        for j, other in enumerate(cleaned):
            if i == j:
                continue
            distances.append(1.0 - SequenceMatcher(None, cand, other).ratio())
        score = (sum(distances) / max(1, len(distances))) + text_quality_penalty(cand) + i * 1e-4
        if score < best_score:
            best_score = score
            best_text = cand
    return best_text



In [ ]:
def get_yolo_names(yolo_model):
    names = getattr(yolo_model, "names", None)
    if names is None and hasattr(yolo_model, "model"):
        names = getattr(yolo_model.model, "names", None)
    return names or {}


def get_class_name(names, cls_id):
    if isinstance(names, dict):
        return names.get(cls_id, names.get(str(cls_id), "handwritten"))
    if isinstance(names, (list, tuple)) and 0 <= cls_id < len(names):
        return names[cls_id]
    return "handwritten"


def load_yolo_model(device):
    print(f"Loading YOLO on {device}: {yolo_weights}", flush=True)
    yolo_model = YOLOv10(str(yolo_weights))
    try:
        yolo_model.to(device)
    except Exception as e:
        print("YOLO .to(device) skipped:", e, flush=True)
    return yolo_model


def dedupe_yolo_regions(regions):
    kept = []
    for region in sorted(regions, key=lambda r: float(r.get("_score", 0.0)), reverse=True):
        duplicate = any(iou(region["bbox"], old["bbox"]) > YOLO_DEDUP_IOU for old in kept)
        if not duplicate:
            kept.append(region)
    return sort_regions([strip_internal_fields(r) for r in kept])


def postprocess_yolo_result(result, img_w, img_h, names):
    boxes = []
    scores = []
    labels = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        boxes.append([x1, y1, x2, y2])
        scores.append(float(box.conf[0]))
        labels.append(int(box.cls[0]))

    if not boxes:
        return []

    boxes_t = torch.tensor(boxes, dtype=torch.float32)
    scores_t = torch.tensor(scores, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.int64)

    keep_indices = []
    for cls_id in sorted(set(labels)):
        cls_mask = labels_t == cls_id
        cls_indices = torch.nonzero(cls_mask, as_tuple=True)[0]
        kept = nms(boxes_t[cls_indices], scores_t[cls_indices], YOLO_IOU_NMS)
        keep_indices.extend(cls_indices[kept].tolist())

    regions = []
    for idx in sorted(set(keep_indices)):
        x1, y1, x2, y2 = boxes_t[idx].tolist()
        h = max(1.0, y2 - y1)
        pad_x = YOLO_PAD_SCALE_X * h
        pad_y = YOLO_PAD_SCALE_Y * h
        box = clamp_xyxy([x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y], img_w, img_h)
        if box is None:
            continue
        cls_id = int(labels_t[idx].item())
        rtype = normalize_type(get_class_name(names, cls_id))
        regions.append({"bbox": box, "type": rtype, "text": "", "_score": float(scores_t[idx].item())})

    return dedupe_yolo_regions(regions)


def detect_regions_yolo(yolo_model, image_path, yolo_device):
    results = yolo_model.predict(
        source=str(image_path),
        imgsz=YOLO_IMG_SIZE,
        conf=YOLO_CONF,
        max_det=YOLO_MAX_DET,
        verbose=False,
        device=yolo_device,
    )
    result = results[0]
    if hasattr(result, "orig_shape") and result.orig_shape:
        img_h, img_w = result.orig_shape
    else:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
    return postprocess_yolo_result(result, img_w=img_w, img_h=img_h, names=get_yolo_names(yolo_model))



In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig


def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def adapter_name_from_path(path, index):
    text = str(path).lower()
    match = re.search(r"model(\d+)", text)
    if match:
        return f"oof_v2_m{match.group(1)}"
    return f"oof_v2_{index}"


def get_active_adapter_name(model):
    active = getattr(model, "active_adapter", None)
    if callable(active):
        try:
            return active()
        except Exception:
            return "default"
    return active or "default"


def make_model_load_kwargs(device):
    kwargs = {
        "device_map": {"": device},
        "dtype": TORCH_DTYPE,
        "trust_remote_code": True,
        "attn_implementation": "sdpa",
        "low_cpu_mem_usage": True,
    }
    if MODEL_LOAD_MODE in {"4bit", "qlora", "bnb4", "bitsandbytes"}:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=TORCH_DTYPE,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
    elif MODEL_LOAD_MODE not in {"fp16", "float16", "bf16", "bfloat16"}:
        raise ValueError("HTD_MODEL_LOAD_MODE must be fp16/bf16 or 4bit.")
    return kwargs


def load_qwen_bundle(device):
    print(f"Loading Qwen base on {device}: {model_id}", flush=True)
    print("Model load mode:", MODEL_LOAD_MODE, "dtype:", TORCH_DTYPE, flush=True)
    base = AutoModelForImageTextToText.from_pretrained(model_id, **make_model_load_kwargs(device))

    adapter_names = [adapter_name_from_path(path, idx) for idx, path in enumerate(lora_dirs, start=1)]
    first_name = adapter_names[0]
    print(f"Loading adapter {first_name}: {lora_dirs[0]}", flush=True)
    try:
        model = PeftModel.from_pretrained(base, str(lora_dirs[0]), adapter_name=first_name)
    except TypeError:
        model = PeftModel.from_pretrained(base, str(lora_dirs[0]))
        adapter_names[0] = get_active_adapter_name(model)

    loaded_names = [adapter_names[0]]
    if USE_ALL_LORA_ADAPTERS:
        for name, path in zip(adapter_names[1:], lora_dirs[1:]):
            print(f"Loading adapter {name}: {path}", flush=True)
            try:
                model.load_adapter(str(path), adapter_name=name)
                loaded_names.append(name)
            except Exception as e:
                print(f"Skipped adapter {name}: {e}", flush=True)

    model.set_adapter(loaded_names[0])
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    print("Loaded adapters:", loaded_names, flush=True)
    return {"model": model, "processor": processor, "adapter_names": loaded_names}


def apply_chat_template(processor, messages):
    candidates = [
        {"tokenize": False, "add_generation_prompt": True, "template_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "processor_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "enable_thinking": False},
        {"tokenize": False, "add_generation_prompt": True},
    ]
    for kwargs in candidates:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message=r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*",
                )
                return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            continue
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(model, processor, messages_batch, device, max_new_tokens):
    processor.tokenizer.padding_side = "left"
    texts = [apply_chat_template(processor, m) for m in messages_batch]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    try:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            text_kwargs={"padding": True, "return_tensors": "pt"},
            images_kwargs={"return_tensors": "pt"},
            videos_kwargs={"return_tensors": "pt"},
        )
    except TypeError:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
    inputs = inputs.to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=TORCH_DTYPE):
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    if CLEAR_CUDA_CACHE_EVERY_BATCH:
        torch.cuda.empty_cache()
    return decoded


def generate_batch_with_adapter(bundle, adapter_name, messages_batch, device, max_new_tokens):
    model = bundle["model"]
    if adapter_name:
        model.set_adapter(adapter_name)
    return generate_batch(model, bundle["processor"], messages_batch, device, max_new_tokens)


In [ ]:
if not str(globals().get("TRAIN_PAGE_PROMPT", "")).strip():
    raise FileNotFoundError(
        "Missing training page_prompt. The submission notebook must read rukopys_prompt_config.json "
        "from the OOF v2 end-to-end LoRA final directory. Recheck lora_dirs/HTD_OUTPUT_ROOT."
    )

print("Using original training prompt version:", TRAIN_PROMPT_VERSION)
print("Original training prompt source:", TRAIN_PROMPT_SOURCE)
print("Original training prompt chars:", len(TRAIN_PAGE_PROMPT))


def build_crop_prompt(region_type):
    rtype = normalize_type(region_type)
    return (
        "You are given a crop of exactly one document region detected by YOLO. "
        f"The detected region type is {rtype}. "
        "The bbox and type have already been produced by YOLO. Do not detect regions, do not return bbox/type, "
        "and do not return JSON or Markdown. Apply the original end-to-end training prompt below for all OCR, "
        "text-field, document-context, type-specific, correction, table, formula, image, and graph rules. "
        "Return only the text value for this one detected region. "
        "For image or graph regions with no readable text, return an empty string.\n\n"
        "Original end-to-end training prompt used for this LoRA:\n"
        + TRAIN_PAGE_PROMPT
    )


CROP_PROMPTS = {rtype: build_crop_prompt(rtype) for rtype in sorted(VALID_TYPES)}
CROP_PROMPTS["default"] = build_crop_prompt("handwritten")


def resize_to_pixel_budget(img, max_pixels):
    w, h = img.size
    total = max(1, w * h)
    if total <= max_pixels:
        return img
    scale = (max_pixels / total) ** 0.5
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


def crop_image(image_path, bbox):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = bbox
        pad = int(round(max(x2 - x1, y2 - y1) * CROP_PAD_RATIO))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        return resize_to_pixel_budget(img.crop((x1, y1, x2, y2)), MAX_PIXELS_CROP)


def should_crop_ocr(region):
    if CROP_OCR_MODE == "none":
        return False
    if region.get("type") not in TEXT_TYPES:
        return False
    if CROP_OCR_MODE == "all_text":
        return True
    text = str(region.get("text") or "")
    return (not text.strip()) or len(text) < 4 or len(text) > 160


def crop_messages(image_path, region):
    rtype = normalize_type(region.get("type"))
    prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop_image(image_path, region["bbox"]), "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": prompt},
            ],
        }
    ]


def crop_area_for_sort(region):
    x1, y1, x2, y2 = region.get("bbox", [0, 0, 0, 0])
    return max(1, int(x2 - x1) * int(y2 - y1))


def generate_ocr_texts(bundle, messages_batch, device):
    adapter_names = bundle["adapter_names"]
    if USE_ALL_LORA_ADAPTERS and len(adapter_names) > 1:
        per_adapter = []
        for adapter_name in adapter_names:
            t0 = time.time()
            outs = generate_batch_with_adapter(bundle, adapter_name, messages_batch, device, MAX_NEW_TOKENS_CROP)
            if PROFILE_OCR:
                print(f"    adapter={adapter_name} crop_batch={len(messages_batch)} generate={time.time() - t0:.1f}s", flush=True)
            per_adapter.append([clean_crop_text(x) for x in outs])
        merged = []
        for idx in range(len(messages_batch)):
            merged.append(choose_consensus_text([adapter_outputs[idx] for adapter_outputs in per_adapter]))
        return merged
    t0 = time.time()
    outs = generate_batch_with_adapter(bundle, adapter_names[0], messages_batch, device, MAX_NEW_TOKENS_CROP)
    if PROFILE_OCR:
        print(f"    adapter={adapter_names[0]} crop_batch={len(messages_batch)} generate={time.time() - t0:.1f}s", flush=True)
    return [clean_crop_text(x) for x in outs]


def ocr_regions(bundle, image_path, regions, device):
    crop_indices = [i for i, r in enumerate(regions) if should_crop_ocr(r)]
    if SORT_CROPS_BY_AREA:
        crop_indices.sort(key=lambda i: crop_area_for_sort(regions[i]))

    pos = 0
    batch_size = max(1, int(CROP_BATCH_SIZE))
    if PROFILE_OCR:
        prompt_chars = len(CROP_PROMPTS.get("handwritten", CROP_PROMPTS["default"]))
        print(
            f"  OCR start crops={len(crop_indices)}/{len(regions)} batch_size={batch_size} "
            f"adapters={len(bundle['adapter_names'])} max_pixels={MAX_PIXELS_CROP} "
            f"max_new_tokens={MAX_NEW_TOKENS_CROP} prompt_chars={prompt_chars}",
            flush=True,
        )
    while pos < len(crop_indices):
        batch_indices = crop_indices[pos:pos + batch_size]
        msgs = [crop_messages(image_path, regions[i]) for i in batch_indices]
        batch_t0 = time.time()
        try:
            texts = generate_ocr_texts(bundle, msgs, device)
            if PROFILE_OCR:
                print(
                    f"  OCR batch {pos + 1}-{pos + len(batch_indices)}/{len(crop_indices)} "
                    f"size={len(batch_indices)} total={time.time() - batch_t0:.1f}s",
                    flush=True,
                )
        except torch.cuda.OutOfMemoryError as e:
            torch.cuda.empty_cache()
            gc.collect()
            if batch_size > 1:
                new_batch = max(1, batch_size // 2)
                print(f"Crop OCR OOM at batch_size={batch_size}; retrying with batch_size={new_batch}", flush=True)
                batch_size = new_batch
                continue
            print("Crop OCR failed at batch_size=1 with OOM:", e, flush=True)
            pos += 1
            continue
        except Exception as e:
            print("Crop OCR batch failed:", e, flush=True)
            torch.cuda.empty_cache()
            gc.collect()
            if batch_size > 1:
                new_batch = max(1, batch_size // 2)
                print(f"Retrying failed crop batch with batch_size={new_batch}", flush=True)
                batch_size = new_batch
                continue
            pos += 1
            continue

        for idx, text in zip(batch_indices, texts):
            if text:
                regions[idx]["text"] = text
        pos += len(batch_indices)
    return sort_regions([strip_internal_fields(r) for r in regions])


def infer_one_image(bundle, yolo_model, image_path, device, yolo_device):
    image_t0 = time.time()
    yolo_t0 = time.time()
    regions = detect_regions_yolo(yolo_model, image_path, yolo_device)
    if PROFILE_OCR:
        print(f"  YOLO regions={len(regions)} time={time.time() - yolo_t0:.1f}s image={Path(image_path).name}", flush=True)
    if not regions:
        return []
    ocr_t0 = time.time()
    regions = ocr_regions(bundle, image_path, regions, device)
    if PROFILE_OCR:
        print(f"  OCR done time={time.time() - ocr_t0:.1f}s image_total={time.time() - image_t0:.1f}s", flush=True)
    return sort_regions(regions)



In [ ]:
import multiprocessing as mp


def format_duration(seconds):
    if seconds is None or seconds <= 0:
        return "--:--"
    seconds = int(round(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def progress_bar(done, total, width=PROGRESS_BAR_WIDTH):
    ratio = done / max(1, total)
    filled = min(width, max(0, int(round(width * ratio))))
    return "#" * filled + " " * (width - filled)


def print_progress_line(gpu_id, done, total, run_done, elapsed, region_count, image_name):
    pct = 100.0 * done / max(1, total)
    speed = run_done / elapsed if run_done > 0 and elapsed > 0 else 0.0
    sec_per_img = elapsed / run_done if run_done > 0 else 0.0
    remaining = max(0, total - done)
    eta = remaining / speed if speed > 0 else None
    bar = progress_bar(done, total)
    last = str(image_name or "")[:18]
    if speed > 0:
        timing = f"{format_duration(elapsed)}<{format_duration(eta)}, {sec_per_img:.2f}s/img, speed={speed:.2f} img/s"
    else:
        timing = "resume, speed=-- img/s"
    print(
        f"GPU {gpu_id}: {pct:3.0f}%|{bar}| {done}/{total} "
        f"[{timing}, regions={region_count}, last={last}]",
        flush=True,
    )


def record_image_name(rec):
    return Path(rec.get("file_name") or rec.get("image") or "").name


def worker_process(gpu_id, records, output_csv):
    suppress_transformers_noise()
    device = f"cuda:{gpu_id}"
    total_records = len(records)
    print(f"[GPU {gpu_id}] loading Qwen adapters and YOLO for {total_records} images", flush=True)
    bundle = load_qwen_bundle(device)
    yolo_model = load_yolo_model(device)
    print(f"[GPU {gpu_id}] models loaded; starting YOLO + OOF v2 crop OCR", flush=True)

    done = set()
    results = []
    if RESUME_PARTIALS and Path(output_csv).exists():
        try:
            old = pd.read_csv(output_csv)
            old = old.drop_duplicates(subset=["image"], keep="last")
            done = set(old["image"].tolist())
            results = old.to_dict("records")
            print(f"[GPU {gpu_id}] resumed {len(done)}/{total_records} rows from {output_csv}", flush=True)
        except Exception as e:
            print(f"[GPU {gpu_id}] could not read checkpoint: {e}", flush=True)

    print_progress_line(gpu_id, len(done), total_records, 0, 0.0, 0, "resumed" if done else "start")

    pbar = None
    if USE_TQDM_PROGRESS:
        pbar = tqdm(
            total=total_records,
            initial=len(done),
            desc=f"GPU {gpu_id}",
            position=gpu_id,
            leave=True,
            dynamic_ncols=True,
            smoothing=0.05,
            unit="img",
            mininterval=1.0,
            maxinterval=10.0,
            file=sys.stdout,
        )
    run_start = time.time()
    run_done = 0

    for rec in records:
        image_name = record_image_name(rec)
        if image_name in done:
            continue
        image_path = resolve_image_path(dataset_root, IMAGE_SPLIT, rec.get("file_name") or image_name)
        try:
            regions = infer_one_image(bundle, yolo_model, image_path, device, gpu_id)
        except Exception as e:
            print(f"[GPU {gpu_id}] failed {image_name}: {e}", flush=True)
            regions = []
            torch.cuda.empty_cache()
            gc.collect()
        results.append({"image": image_name, "regions": json.dumps(regions, ensure_ascii=False)})
        done.add(image_name)
        run_done += 1
        elapsed = max(1e-6, time.time() - run_start)
        if pbar is not None:
            pbar.set_postfix_str(
                f"speed={run_done / elapsed:.2f} img/s, last_regions={len(regions)}, last={image_name[:12]}"
            )
            pbar.update(1)
        if run_done % PROGRESS_LOG_EVERY == 0 or len(done) == total_records:
            print_progress_line(gpu_id, len(done), total_records, run_done, elapsed, len(regions), image_name)

        if len(results) % CHECKPOINT_EVERY == 0:
            tmp = output_csv + ".tmp"
            pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
            os.replace(tmp, output_csv)
            if pbar is not None:
                pbar.set_postfix_str(
                    f"speed={run_done / elapsed:.2f} img/s, last_regions={len(regions)}, ckpt={len(results)}"
                )

    if pbar is not None:
        pbar.close()

    tmp = output_csv + ".tmp"
    pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
    os.replace(tmp, output_csv)
    print(f"[GPU {gpu_id}] done {len(done)}/{total_records}; saved {output_csv}", flush=True)


def detect_num_gpus():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"]).decode("utf-8").strip()
        return max(1, len([x for x in out.splitlines() if x.strip()]))
    except Exception:
        return max(1, torch.cuda.device_count())


def select_num_gpus():
    detected = detect_num_gpus()
    requested = max(1, int(REQUESTED_NUM_GPUS or detected))
    selected = min(detected, requested)
    print(f"GPUs detected={detected}, requested={requested}, using={selected}", flush=True)
    if REQUIRE_REQUESTED_GPUS and detected < requested:
        raise RuntimeError(f"Requested {requested} GPUs but only detected {detected}.")
    if detected < requested:
        print("Warning: fewer GPUs than requested; continuing with the visible GPU count.", flush=True)
    return selected


def split_records_for_gpus(records, num_gpus):
    chunk_size = math.ceil(len(records) / num_gpus)
    return [records[i * chunk_size:(i + 1) * chunk_size] for i in range(num_gpus)]


if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for Qwen3-VL 8B inference.")

if os.name != "nt":
    mp.set_start_method("fork", force=True)
num_gpus = select_num_gpus()
chunks = split_records_for_gpus(test_records, num_gpus)
for gpu_id, chunk in enumerate(chunks):
    print(f"GPU {gpu_id}: assigned {len(chunk)} images", flush=True)

processes = []
partials = []
if num_gpus == 1:
    output_csv = f"{HYBRID_PARTIAL_PREFIX}0.csv"
    partials.append(output_csv)
    print("Single GPU detected; running worker inline for clearer model-load logs.", flush=True)
    worker_process(0, chunks[0], output_csv)
else:
    for gpu_id, chunk in enumerate(chunks):
        if not chunk:
            continue
        output_csv = f"{HYBRID_PARTIAL_PREFIX}{gpu_id}.csv"
        partials.append(output_csv)
        p = mp.Process(target=worker_process, args=(gpu_id, chunk, output_csv))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    bad_exitcodes = [p.exitcode for p in processes if p.exitcode not in (0, None)]
    if bad_exitcodes:
        raise RuntimeError(f"One or more workers failed with exit codes: {bad_exitcodes}")

frames = []
for path in partials:
    if Path(path).exists():
        frame = pd.read_csv(path)
        print(f"Partial {path}: rows={len(frame)}", flush=True)
        frames.append(frame)
if not frames:
    raise RuntimeError("No partial outputs were created.")

final = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["image"], keep="last")
order = [record_image_name(r) for r in test_records]
final = final.set_index("image").reindex(order).reset_index()
final["regions"] = final["regions"].fillna("[]")
final.to_csv(OUTPUT_CSV, index=False)
print("Wrote", OUTPUT_CSV, "rows=", len(final), flush=True)
final.head()



In [ ]:
df = pd.read_csv(OUTPUT_CSV)
assert list(df.columns) == ["image", "regions"], df.columns
assert len(df) == len(test_records), (len(df), len(test_records))

bad = []
region_counts = []
for row in df.itertuples(index=False):
    try:
        parsed = json.loads(row.regions)
        assert isinstance(parsed, list)
        region_counts.append(len(parsed))
        for item in parsed:
            assert set(item.keys()) == {"bbox", "type", "text"}, item.keys()
            assert isinstance(item["bbox"], list) and len(item["bbox"]) == 4
            assert all(isinstance(x, int) for x in item["bbox"]), item["bbox"]
            assert item["type"] in VALID_TYPES, item["type"]
            assert isinstance(item["text"], str)
    except Exception as e:
        bad.append((row.image, str(e)))
        if len(bad) >= 5:
            break

print("Bad rows:", bad[:5])
print("Images:", len(df))
print("Total regions:", sum(region_counts))
print("Avg regions/page:", round(sum(region_counts) / max(1, len(region_counts)), 2))
print("Ready:", OUTPUT_CSV)

df.head()
